In [3]:
from pyspark.sql import functions as F

# Bronze-Daten laden
df = spark.table("bronze.vehicle_diagnostics")

# Datenqualitätsprüfungen
total_rows = df.count()

null_ids = df.filter(
    F.col("TestID").isNull()
).count()

duplicate_ids = (
    df.groupBy("TestID")
      .count()
      .filter(F.col("count") > 1)
      .count()
)

invalid_results = df.filter(
    F.col("TestResult").isNull() |
    (~F.upper(F.trim(F.col("TestResult")))
      .isin("PASS", "FAIL"))
).count()

# Ergebnisse ausgeben
print("Datensätze:", total_rows)
print("Fehlende TestIDs:", null_ids)
print("Doppelte TestIDs:", duplicate_ids)
print("Ungültige Testergebnisse:", invalid_results)

if null_ids == 0 and duplicate_ids == 0 and invalid_results == 0:
    print("Data Quality Status: PASS")
else:
    print("Data Quality Status: FAIL")


StatementMeta(, ebdb48da-99ca-459a-b16c-aaaf2b279472, 5, Finished, Available, Finished, False)

Datensätze: 5
Fehlende TestIDs: 0
Doppelte TestIDs: 0
Ungültige Testergebnisse: 0
Data Quality Status: PASS


In [4]:
from pyspark.sql import functions as F

# Ergebnisse des aktuellen Prüflaufs
quality_status = (
    "PASS"
    if null_ids == 0
    and duplicate_ids == 0
    and invalid_results == 0
    else "FAIL"
)

# Protokolleintrag als DataFrame erzeugen
quality_data = [(
    int(total_rows),
    int(null_ids),
    int(duplicate_ids),
    int(invalid_results),
    quality_status
)]

quality_columns = [
    "TotalRows",
    "MissingTestIDs",
    "DuplicateTestIDs",
    "InvalidResults",
    "QualityStatus"
]

df_quality = spark.createDataFrame(
    quality_data,
    quality_columns
).withColumn(
    "CheckedAt",
    F.current_timestamp()
)

# Qualitätsergebnisse dauerhaft speichern
df_quality.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("dbo.data_quality_log")

print("Data-Quality-Ergebnis gespeichert!")

display(df_quality)


StatementMeta(, ebdb48da-99ca-459a-b16c-aaaf2b279472, 6, Finished, Available, Finished, False)

Data-Quality-Ergebnis gespeichert!


SynapseWidget(Synapse.DataFrame, edde8f55-718d-41b0-8f13-960f07bc281e)

In [6]:
# Data Quality Gate
# Bei fehlerhaften Daten wird die Pipeline gestoppt.

if quality_status == "FAIL":
    raise ValueError(
        "DATA QUALITY FAILED: "
        f"MissingTestIDs={null_ids}, "
        f"DuplicateTestIDs={duplicate_ids}, "
        f"InvalidResults={invalid_results}"
    )

print("DATA QUALITY PASSED - Silver kann starten.")


StatementMeta(, ebdb48da-99ca-459a-b16c-aaaf2b279472, 8, Finished, Available, Finished, False)

DATA QUALITY PASSED - Silver kann starten.
